# Wind Generation Reliability Analysis

This notebook analyses the reliability and statistical distribution of UK national
wind power generation using actual half-hourly FUELHH data.

**Period:** January 2025 – present  
**Key output:** Reliability curve + P10 reliable capacity estimate

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone

from data_fetcher import fetch_actuals

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Libraries loaded.')

## 1. Fetch All Actuals (Jan 2025 – Present)

In [ ]:
# Fetch in monthly chunks to avoid timeout
FROM_DT = datetime(2025, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
TO_DT   = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

print(f'Fetching actuals from {FROM_DT.date()} to {TO_DT.date()}...')

# Chunk by month to avoid 7-day API limit when called directly
all_actuals = []
current = FROM_DT
while current < TO_DT:
    chunk_end = min(current + pd.Timedelta(days=6), TO_DT)
    print(f'  {current.date()} → {chunk_end.date()}')
    chunk = fetch_actuals(current, chunk_end)
    if not chunk.empty:
        all_actuals.append(chunk)
    current = chunk_end

actuals = pd.concat(all_actuals).drop_duplicates('startTime').sort_values('startTime').reset_index(drop=True)
print(f'\nTotal records: {len(actuals)}')
print(f'Date range: {actuals.startTime.min()} → {actuals.startTime.max()}')
actuals.head()

## 2. Summary Statistics

In [ ]:
gen = actuals['actualMW']

print('=== UK Wind Generation Summary (MW) ===')
print(f'Count:       {len(gen):,} half-hour periods')
print(f'Mean:        {gen.mean():,.0f} MW')
print(f'Std dev:     {gen.std():,.0f} MW')
print(f'Min:         {gen.min():,.0f} MW')
print(f'P10:         {gen.quantile(0.10):,.0f} MW  ← reliable capacity estimate')
print(f'P25:         {gen.quantile(0.25):,.0f} MW')
print(f'Median:      {gen.median():,.0f} MW')
print(f'P75:         {gen.quantile(0.75):,.0f} MW')
print(f'P90:         {gen.quantile(0.90):,.0f} MW')
print(f'Max:         {gen.max():,.0f} MW')
print(f'\nCapacity factor (median / assumed 28 GW installed):')
print(f'  Median CF: {gen.median()/28000*100:.1f}%')
print(f'  Mean CF:   {gen.mean()/28000*100:.1f}%')

## 3. Full Generation Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actuals['startTime'], actuals['actualMW'], linewidth=0.6, color='steelblue', alpha=0.8)
ax.axhline(gen.quantile(0.10), color='tomato', linestyle='--', linewidth=1.5,
           label=f'P10 = {gen.quantile(0.10):.0f} MW (reliable capacity)')
ax.axhline(gen.mean(), color='darkorange', linestyle=':', linewidth=1.5,
           label=f'Mean = {gen.mean():.0f} MW')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Generation (MW)', fontsize=12)
ax.set_title('UK Wind Generation Time Series (Jan 2025 – present)', fontsize=13)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('wind_time_series.png', dpi=150)
plt.show()

## 4. Generation Histogram with Percentile Lines

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(gen, bins=80, color='steelblue', edgecolor='white', alpha=0.75)

percentiles = [(10, 'P10', 'tomato'), (50, 'P50', 'darkorange'), (90, 'P90', 'green')]
for pct, label, color in percentiles:
    val = gen.quantile(pct/100)
    ax.axvline(val, color=color, linestyle='--', linewidth=1.8,
               label=f'{label} = {val:,.0f} MW')

ax.set_xlabel('Generation (MW)', fontsize=12)
ax.set_ylabel('Count (half-hours)', fontsize=12)
ax.set_title('UK Wind Generation Distribution', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('wind_generation_histogram.png', dpi=150)
plt.show()

## 5. Load Duration Curve

In [ ]:
sorted_gen = np.sort(gen.values)[::-1]  # descending
pct_time = np.linspace(0, 100, len(sorted_gen))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pct_time, sorted_gen, color='steelblue', linewidth=1.8)
ax.axvline(10, color='tomato', linestyle='--', linewidth=1.5,
           label=f'10% of time → {np.percentile(gen, 90):,.0f} MW')
ax.axvline(90, color='green', linestyle='--', linewidth=1.5,
           label=f'90% of time → {np.percentile(gen, 10):,.0f} MW')
ax.set_xlabel('% of Time Exceeded', fontsize=12)
ax.set_ylabel('Generation (MW)', fontsize=12)
ax.set_title('Wind Generation Load Duration Curve', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('wind_load_duration_curve.png', dpi=150)
plt.show()

## 6. Reliability Function

For each threshold T (MW), what fraction of half-hours has generation ≥ T?

In [ ]:
thresholds = np.arange(0, gen.max() + 500, 250)
reliability = [(gen >= t).mean() * 100 for t in thresholds]

p10_mw = gen.quantile(0.10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, reliability, color='steelblue', linewidth=2)
ax.axhline(90, color='tomato', linestyle='--', linewidth=1.5, label='90% reliability')
ax.axvline(p10_mw, color='tomato', linestyle=':', linewidth=1.5,
           label=f'P10 = {p10_mw:,.0f} MW (reliable at 90%)')
ax.fill_between(thresholds, reliability, alpha=0.12, color='steelblue')
ax.set_xlabel('Generation Threshold (MW)', fontsize=12)
ax.set_ylabel('% of Time Generation ≥ Threshold', fontsize=12)
ax.set_title('Wind Generation Reliability Curve', fontsize=13)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('wind_reliability_curve.png', dpi=150)
plt.show()
print(f'\nP10 reliable capacity: {p10_mw:,.0f} MW')
print(f'(Wind generates at or above this level 90% of the time)')

## 7. Monthly / Seasonal Analysis

In [ ]:
actuals['month'] = actuals['startTime'].dt.month
actuals['month_name'] = actuals['startTime'].dt.strftime('%b %Y')

monthly_groups = actuals.groupby('month_name')['actualMW']
unique_months = actuals.sort_values('startTime')['month_name'].unique()

fig, ax = plt.subplots(figsize=(max(10, len(unique_months) * 1.2), 5))
bp_data = [monthly_groups.get_group(m).values for m in unique_months]
bp = ax.boxplot(bp_data, patch_artist=True, medianprops=dict(color='white', linewidth=2))

colors = plt.cm.tab10(np.linspace(0, 1, len(unique_months)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(unique_months, rotation=30, ha='right')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Generation (MW)', fontsize=12)
ax.set_title('Monthly Wind Generation Distribution', fontsize=13)

# P10 reference line
ax.axhline(p10_mw, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Overall P10 = {p10_mw:,.0f} MW')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('wind_monthly_boxplot.png', dpi=150)
plt.show()

## 8. Recommendation: P10 as Reliable Capacity

### Derivation

The **P10 generation level** is defined as the 10th percentile of all half-hourly
generation readings. By definition, actual wind generation equals or exceeds this
value **90% of the time** — it is the "firm" capacity that can be relied upon
with high confidence.

### Justification

1. **Load duration curve**: The curve shows that generation drops below the P10 level
   only in the left tail of the distribution. The 90% exceedance threshold provides a
   pragmatic balance between conservatism and capacity value.

2. **Reliability curve**: The reliability function confirms that the P10 value corresponds
   to exactly the 90% reliability threshold. Operators can plan firm generation commitments
   up to this level without requiring backup in 90% of half-hours.

3. **Comparison to mean**: The mean generation is significantly higher than P10, confirming
   that wind is highly variable. Using the mean as a planning figure would over-estimate
   reliable capacity in the majority of situations.

4. **Seasonal variation**: The monthly boxplots show that winter months (Jan–Feb) tend to
   have higher median and P10 generation than summer months, consistent with the UK's
   seasonal wind patterns. Annual P10 values are conservative in winter but appropriate
   for year-round planning.

### Recommendation

> **Use P10 = {:.0f} MW as the reliable wind capacity figure for operational planning.**
> This represents the generation level that can be counted on with 90% confidence across
> all time periods in the analysis window. For seasonal-specific planning, use monthly
> P10 values from the boxplot analysis above.

In [ ]:
print(f'Final recommendation:')
print(f'  Reliable capacity (P10): {p10_mw:,.0f} MW')
print(f'  Mean generation:         {gen.mean():,.0f} MW')
print(f'  Peak generation:         {gen.max():,.0f} MW')
print(f'  P10 / Peak ratio:        {p10_mw/gen.max()*100:.1f}%')
print()
print('Monthly P10 values:')
for month in unique_months:
    m_gen = monthly_groups.get_group(month)
    print(f'  {month}: P10 = {m_gen.quantile(0.10):,.0f} MW  (n={len(m_gen):,})')